# Jodhpur House Price Prediction
## 1. Data Cleaning & Exploratory Data Analysis (EDA)

This notebook processes the raw data, handles missing values and outliers, visualizes the distribution, and exports a cleaned dataset for model training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

### Load Raw Data

In [ ]:
# Load the dirty dataset
df = pd.read_csv('jodhpur_housing_dirty.csv')
print(f"Initial shape: {df.shape}")
df.head()

### Data Cleaning
Remove unnecessary columns and handle missing values.

In [ ]:
# 1. Drop unwanted columns
cols_to_drop = ['Distance_Airport', 'Distance_Railway_Station', 'Distance_City_Center', 'Furnishing_Status', 'Total_Floors', 'Floor_No']
df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)

# 2. Convert Area_Size to numeric
if 'Area_Size' in df.columns and df['Area_Size'].dtype == 'object':
    df['Area_Size'] = pd.to_numeric(df['Area_Size'], errors='coerce')

# 3. Handle missing values (Median for numerical, Mode for categorical)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

print(f"Missing values remaining: {df.isnull().sum().sum()}")

### Handle Outliers (IQR Method)

In [ ]:
# Cap outliers using 1.5 * IQR
continuous_cols = ['Area_Size', 'Price', 'Property_Age']

for col in continuous_cols:
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
        df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

print("Outliers capped successfully.")

### Format Categorical Data

In [ ]:
# Convert Yes/No amenities to 1/0
yes_no_columns = ['Gym', 'Swimming_Pool', 'Market', 'School', 'Hospital', 
                  'Park', 'Mall', 'Security', 'Lift', 'Parking', 'Power_Backup']

for col in yes_no_columns:
    if col in df.columns:
        df[col] = df[col].map({'Yes': 1, 'No': 0, 1: 1, 0: 0})
        df[col] = df[col].fillna(0).astype(int)
        
df.head()

### Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['Price'], bins=50, kde=True)
plt.title('Distribution of Property Prices')
plt.xlabel('Price')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='Area_Name', y='Price', data=df)
plt.title('Price Distribution by Area')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Correlation matrix for numerical features
numeric_df = df.select_dtypes(include=['int64', 'float64', 'int32'])
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

### Save Cleaned Dataset

In [ ]:
df.to_csv('jodhpur_housing_cleaned.csv', index=False)
print(f"Cleaned dataset saved. Final shape: {df.shape}")